# Gmail Janitor - AI-Powered Email Cleanup

**Course:** Generative AI - Spring 2026  
**Project:** Text Extraction & Classification with LLMs  

---

This notebook demonstrates using **Gemini 2.5 Flash** to classify Gmail emails as marketing junk vs. important correspondence. The LLM performs Chain-of-Thought reasoning on email metadata to produce structured JSON decisions.

### Workflow
1. Authenticate with Gmail API (OAuth2)
2. Search for emails by keyword
3. Extract email metadata (From, Subject, Date, Snippet)
4. Send each email to Gemini for classification
5. Visualize the results
6. Optionally trash confirmed junk

In [ ]:
import os
import sys
import time
import json
from pathlib import Path
from typing import Literal

import yaml
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google import genai
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

print("All imports loaded successfully.")

In [ ]:
# Configuration
SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]
PROJECT_DIR = Path.cwd()
CREDENTIALS_FILE = PROJECT_DIR / "credentials.json"
TOKEN_FILE = PROJECT_DIR / "token.json"
PROMPTS_FILE = PROJECT_DIR / "prompts.yml"
GEMINI_MODEL = "gemini-2.5-flash"
RATE_LIMIT_DELAY = 4.0

# Load environment variables
load_dotenv(PROJECT_DIR / ".env")

# Load prompts
with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompts = yaml.safe_load(f)

print("Configuration loaded.")
print(f"Project directory: {PROJECT_DIR}")

In [ ]:
# Pydantic model for structured Gemini output
class EmailClassification(BaseModel):
    reasoning: str = Field(
        description="Step-by-step reasoning for the classification decision"
    )
    decision: Literal["trash", "keep"] = Field(
        description="Whether to trash or keep the email"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence score from 0.0 to 1.0",
    )

print("Schema:")
print(json.dumps(EmailClassification.model_json_schema(), indent=2))

## Step 1: Authenticate with Gmail

Running this cell will open your browser for Google OAuth consent (first time only). After that, `token.json` is saved locally for reuse.

In [ ]:
creds = None

if TOKEN_FILE.exists():
    creds = Credentials.from_authorized_user_file(str(TOKEN_FILE), SCOPES)

if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        try:
            creds.refresh(Request())
        except Exception:
            creds = None

    if not creds:
        flow = InstalledAppFlow.from_client_secrets_file(
            str(CREDENTIALS_FILE), SCOPES
        )
        creds = flow.run_local_server(port=0)

    with open(TOKEN_FILE, "w") as token:
        token.write(creds.to_json())

service = build("gmail", "v1", credentials=creds)
print("Gmail authenticated successfully!")

In [ ]:
# Initialize Gemini client
api_key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
if not api_key:
    api_key = input("Enter your Gemini API key: ").strip()

gemini_client = genai.Client(api_key=api_key)
print("Gemini client initialized.")

## Step 2: Search for Emails by Keyword

Enter comma-separated keywords (company names, sender names, etc.). The script searches `from:` and `subject:` fields.

In [ ]:
keywords_input = input("Enter keywords (comma-separated): ")
keywords = [kw.strip() for kw in keywords_input.split(",") if kw.strip()]

# Build Gmail search query
query_parts = []
for kw in keywords:
    query_parts.append(f"from:({kw})")
    query_parts.append(f"subject:({kw})")
query = " OR ".join(query_parts)

print(f"Keywords: {keywords}")
print(f"Gmail query: {query}")

In [ ]:
# Fetch matching email IDs (with pagination)
all_message_ids = []
page_token = None

while True:
    results = (
        service.users()
        .messages()
        .list(userId="me", q=query, maxResults=100, pageToken=page_token)
        .execute()
    )
    messages = results.get("messages", [])
    all_message_ids.extend(messages)

    page_token = results.get("nextPageToken")
    if not page_token:
        break

print(f"Found {len(all_message_ids)} matching emails.")

# Cap at 50 for demo purposes
all_message_ids = all_message_ids[:50]
print(f"Processing first {len(all_message_ids)} emails.")

In [ ]:
# Fetch metadata for each email
emails = []
for i, msg_stub in enumerate(all_message_ids):
    msg = (
        service.users()
        .messages()
        .get(
            userId="me",
            id=msg_stub["id"],
            format="metadata",
            metadataHeaders=["From", "Subject", "Date"],
        )
        .execute()
    )
    headers = {
        h["name"]: h["value"]
        for h in msg.get("payload", {}).get("headers", [])
    }
    emails.append(
        {
            "id": msg["id"],
            "sender": headers.get("From", "Unknown"),
            "subject": headers.get("Subject", "(no subject)"),
            "date": headers.get("Date", "Unknown"),
            "snippet": msg.get("snippet", ""),
        }
    )

df_emails = pd.DataFrame(emails)
print(f"Retrieved metadata for {len(emails)} emails.\n")
df_emails[["sender", "subject", "date"]].head(20)

## Step 3: Prompt Engineering

We use **Chain-of-Thought (CoT)** prompting to force the LLM to reason step-by-step before deciding. The `reasoning` field comes first in the schema so the model generates its thought process *before* committing to a decision.

### System Prompt
Defines the assistant's role, conservative bias (keep when uncertain), and the step-by-step thinking framework.

### Classification Prompt
Fills in the email metadata and asks for a structured JSON response with `reasoning`, `decision`, and `confidence`.

### Structured Output
We use Gemini's `response_json_schema` parameter with a Pydantic model to **guarantee** valid JSON output every time — no regex parsing needed.

In [ ]:
# Display the prompts being used
print("=== SYSTEM PROMPT ===")
print(prompts["system_prompt"])
print("\n=== CLASSIFICATION PROMPT TEMPLATE ===")
print(prompts["classification_prompt"])

## Step 4: Classify Emails with Gemini 2.5 Flash

Each email is sent individually to the model. A 4-second delay between calls stays within the free-tier rate limit (15 RPM).

In [ ]:
results = []
total = len(emails)

for i, email in enumerate(emails):
    print(f"Classifying {i + 1}/{total}: {email['subject'][:60]}...")

    user_prompt = prompts["classification_prompt"].format(
        sender=email["sender"],
        subject=email["subject"],
        date=email["date"],
        snippet=email["snippet"],
    )

    try:
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_prompt,
            config={
                "system_instruction": prompts["system_prompt"],
                "response_mime_type": "application/json",
                "response_json_schema": EmailClassification.model_json_schema(),
                "temperature": 0.2,
            },
        )
        classification = EmailClassification.model_validate_json(response.text)
    except Exception as e:
        print(f"  Error: {e}")
        classification = EmailClassification(
            reasoning=f"Classification failed: {e}. Defaulting to keep.",
            decision="keep",
            confidence=0.0,
        )

    results.append(
        {
            "id": email["id"],
            "sender": email["sender"],
            "subject": email["subject"],
            "date": email["date"],
            "snippet": email["snippet"],
            "decision": classification.decision,
            "confidence": classification.confidence,
            "reasoning": classification.reasoning,
        }
    )

    if i < total - 1:
        time.sleep(RATE_LIMIT_DELAY)

df_results = pd.DataFrame(results)
print(f"\nClassification complete. {len(results)} emails processed.")

## Step 5: Visualize Classification Results

In [ ]:
# Decision distribution - Pie Chart
decision_counts = df_results["decision"].value_counts()

fig = px.pie(
    values=decision_counts.values,
    names=decision_counts.index,
    title="Email Classification Distribution",
    color=decision_counts.index,
    color_discrete_map={"trash": "#ff6b6b", "keep": "#51cf66"},
)
fig.update_traces(textinfo="percent+label+value")
fig.show()

In [ ]:
# Confidence distribution - Histogram
fig = px.histogram(
    df_results,
    x="confidence",
    color="decision",
    nbins=20,
    title="Classification Confidence Distribution",
    color_discrete_map={"trash": "#ff6b6b", "keep": "#51cf66"},
    barmode="overlay",
    opacity=0.7,
    labels={"confidence": "Confidence Score", "count": "Number of Emails"},
)
fig.show()

In [ ]:
# Confidence by sender domain - Box Plot
df_results["domain"] = df_results["sender"].str.extract(r"@([\w.-]+)")
top_domains = df_results["domain"].value_counts().head(10).index
df_top = df_results[df_results["domain"].isin(top_domains)]

if not df_top.empty:
    fig = px.box(
        df_top,
        x="domain",
        y="confidence",
        color="decision",
        title="Confidence by Top Sender Domains",
        color_discrete_map={"trash": "#ff6b6b", "keep": "#51cf66"},
    )
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("Not enough domain data for box plot.")

In [ ]:
# Full results table with color coding
display_cols = ["subject", "sender", "decision", "confidence", "reasoning"]

styled = (
    df_results[display_cols]
    .style.apply(
        lambda row: [
            "background-color: #ffebee" if row["decision"] == "trash"
            else "background-color: #e8f5e9"
        ] * len(row),
        axis=1,
    )
    .format({"confidence": "{:.0%}"})
)
styled

## Step 6: Execute Trash (Optional)

Run the cell below to move AI-recommended junk emails to Trash. You can choose to trash all, none, or select specific emails by number.

In [ ]:
trash_df = df_results[df_results["decision"] == "trash"]
print(f"{len(trash_df)} emails recommended for trash.\n")

for i, (_, row) in enumerate(trash_df.iterrows(), 1):
    print(f"  {i}. {row['subject'][:60]}  ({row['confidence']:.0%})")

print("\nOptions: 'all', 'none', or comma-separated numbers (e.g., '1,3,5')")
choice = input("Your choice: ").strip().lower()

if choice == "all":
    to_trash = trash_df
elif choice == "none" or choice == "":
    to_trash = trash_df.iloc[0:0]  # empty
    print("Cancelled.")
else:
    try:
        indices = [int(x.strip()) - 1 for x in choice.split(",")]
        to_trash = trash_df.iloc[indices]
    except (ValueError, IndexError):
        to_trash = trash_df.iloc[0:0]
        print("Invalid selection. Cancelled.")

trashed = 0
for _, row in to_trash.iterrows():
    try:
        service.users().messages().trash(userId="me", id=row["id"]).execute()
        trashed += 1
        print(f"  Trashed: {row['subject'][:60]}")
    except HttpError as e:
        print(f"  Failed: {e}")

print(f"\n{trashed} email(s) moved to Trash.")

## Observations & Conclusions

*(Fill in after running the notebook)*

- **Model Accuracy:** How well did Gemini distinguish marketing from important emails?
- **Edge Cases:** Were there any emails the model misclassified? Why?
- **Prompt Engineering Impact:** How did the Chain-of-Thought approach affect reasoning quality?
- **Confidence Scores:** Did low-confidence scores correlate with ambiguous emails?
- **Potential Improvements:** What would you change about the prompts or workflow?